## Importing required libraries and modules

In [1]:
from IPython.display import display
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
from astroquery.vizier import Vizier
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd


## Phase 1: DATA


### 1.1) Querying Data

We use the Tempel et al. (2014) SDSS DR10 group catalog. The query returns several tables:

**Galaxies:** The per-galaxy table (positions, redshifts, magnitudes, and Tempel's group IDs). This is the only table used in this project.

**Groups:** Tempel's per-group properties.

**vNNN_gal / vNNN_gr:**
Tempel's group-finding runs at different line-of-sight linking velocities.

We pull the full galaxies table with row_limit=-1 because we will apply our own volume-limited cuts. Tempel's GroupID and Ngal columns are retained only as ground truth labels for validation; they are never used in network construction.

In [ ]:
# Query labeled galaxy groups from Tempel's SDSS DR10 catalog through Vizier
vizier = Vizier(columns=['GalID','RAJ2000','DEJ2000','zobs','rmag','GroupID','Ngal'], row_limit=-1)
print("Querying Vizier for galaxy cluster data...")
catalogs = vizier.get_catalogs("J/A+A/566/A1")
print("Found these tables: \n")
print(catalogs.keys())

### 1.2) Cosmology and the volume-limited sample

We adopt a flat ΛCDM cosmology with $H_0 = 100\,\mathrm{km\,s^{-1}\,Mpc^{-1}},\,\Omega_m = 0.27$, so all distances are expressed in $h^{-1}\,\mathrm{Mpc}$ throughout.

The Main Galaxy Sample (mgs) is built in three steps:

**Column selection:** keep GalID, ra, dec, z, m, GroupID, Ngal from Tempel. GroupID and Ngal are reserved for validation only.

**Redshift slice:** $0.02 \le z \le 0.06$

The lower bound avoids local peculiar velocity contamination and the upper bound keeps the comoving volume small enough that a single linking length remains physically meaningful. Higher redshifts introduce complications that are beyond the scope of this project.

**Volume-limited cut:** SDSS is a flux-limited survey. This means that faint galaxies at low redshift, which would be undetected at higher redshift, must be excluded to maintain a uniform sample.

The absolute magnitude limit is defined at $z_{\max}$:

$$M_{\text{lim}} \;=\; m_{\text{lim}} \;-\; \mu(z_{\max}),$$

with $m_{\text{lim}} = 17.77$ (SDSS Main Sample). Galaxies fainter than $M_{\text{lim}}$ are discarded. This ensures uniform completeness across the volume.

In [ ]:
# Define the cosmology
cosmo = FlatLambdaCDM(H0=100, Om0=0.27)
# Load the first table (the galaxy list)
df = catalogs[0].to_pandas()

# Retain only the required columns
df = df[['GalID', 'RAJ2000', 'DEJ2000', 'zobs','rmag', 'GroupID', 'Ngal']].copy()

# Rename to standard variable names
df.rename(columns={
    'RAJ2000': 'ra', 
    'DEJ2000': 'dec', 
    'zobs': 'z',
    'rmag': 'm'
}, inplace=True)
# Build the main galaxy sample dataframe (mgs)
# Apply the redshift slice (0.02 <= z <= 0.06)
z_max = 0.06
z_min = 0.02
mgs = df[
    (df['z'] >= z_min) & 
    (df['z'] <= z_max)
    ].copy()
# Exclude galaxies fainter than the absolute magnitude limit at z_max
m_limit = 17.77 # Apparent magnitude limit of SDSS Main Galaxy Sample
M_Limit = m_limit - cosmo.distmod(z_max).value  # Absolute magnitude limit at z_max
mgs['M'] = mgs['m'] - cosmo.distmod(mgs['z']).value # Absolute magnitude for each galaxy
mgs = mgs[mgs['M'] <= M_Limit].copy()
mgs.reset_index(drop=True, inplace=True)

print(f"Number of galaxies before cuts: {len(df)}")
print(f"Number of galaxies after cuts: {len(mgs)}")
print(f"Sample catalog:")
display(mgs.sample(5).style.set_caption("Filtered data"))

### 1.3) Coordinate conversion

The Tempel catalog provides each galaxy as $(d,\,\alpha,\,\delta)$. To build the network, we convert to Cartesian coordinates $(X, Y, Z)$:

$X = d\cos(\delta)\cos(\alpha)$

$Y = d\cos(\delta)\sin(\alpha)$

$Z = d\sin(\delta)$

where $d$ is the comoving distance and ($\alpha$ as ra, $\delta$ as dec) are the equatorial coordinates, with Earth at the origin. The line-of-sight unit vector to galaxy $i$ is $\hat{r}_i = (X,Y,Z)\,/\,|(X,Y,Z)|$, which later permits decomposition into a transverse component $r_\perp$ and a line-of-sight component $r_\parallel$.

In [ ]:
# Calculate comoving distance in h^-1 Mpc
mgs['distance_hinv_mpc'] = cosmo.comoving_distance(mgs['z']).to(u.Mpc).value / cosmo.h

# Summon the skycoord and let it do its magic. (YES! this could be done using numpy but we are astrophysicists =) )
coords = SkyCoord(
    ra=mgs['ra'].values * u.degree,
    dec=mgs['dec'].values * u.degree,
    distance=mgs['distance_hinv_mpc'].values * u.Mpc
    )

# Extract the 3D Cartesian coordinates
mgs['X'] = coords.cartesian.x.value
mgs['Y'] = coords.cartesian.y.value
mgs['Z'] = coords.cartesian.z.value

print("Here are the 3D coordinates:")
display(mgs[['GalID', 'X', 'Y', 'Z', 'GroupID']].sample(5).style.set_caption("3D Coordinates with GroupID"))

________________________________________________________________________________________________
## Phase 2: Building the network

### 2.1) KDTree and the cylindrical linking criterion

Nodes are the galaxies. In Phase 2, we define the conditions under which two nodes are connected.

**Spatial search:** The KDTree on the redshift-space Cartesian coordinates finds all pairs within radius $r$ efficiently.

**Anisotropic linking.** A pure spherical FoF performs poorly in redshift space because virialized galaxies have peculiar velocities of $\sim 300$--$500$ km/s, which displace groups along the line of sight by several Mpc. We therefore use a **cylindrical** linking criterion:

A pair $(i, j)$ is linked if

$$
d_\perp \leq r_\perp \quad \text{and} \quad d_\parallel \leq r_\parallel = S \cdot r_\perp .
$$

where $S$ is the **stretch factor**, currently fixed at $S = 10$ following Tempel et al. (2014). While Tempel et al. fix the stretch factor, they use a redshift-dependent $r_\perp$. To reduce the computational cost, we adopt a fixed $r_\perp$, which degrades performance at the extremes of the redshift slice.

Here $\hat{\ell}_{ij}$ is the unit line-of-sight vector at the midpoint $(\vec{r}_i + \vec{r}_j) / 2$, so

$$
d_\parallel = \left| (\vec{r}_i - \vec{r}_j) \cdot \hat{\ell}_{ij} \right|,
\qquad
d_\perp = \sqrt{ |\vec{r}_i - \vec{r}_j|^2 - d_\parallel^2 }.
$$

The KDTree first returns all pairs inside the bounding sphere with $r_{\max} = \sqrt{r_\perp^2 + r_\parallel^2}$, which are then filtered by the cylinder condition.

In [ ]:
# Build KDTree for spatial searching
coordinates = mgs[['X', 'Y', 'Z']].values
tree = KDTree(coordinates)
number_of_galaxies = len(mgs)

# Stretch factor for the Fingers-of-God effect.
# Typical cluster velocity dispersions correspond to a line-of-sight stretch of 5--8; Tempel et al. (2014) adopt 10.
stretch_factor = 10

# Helper function to find edges using a cylinder
def get_cylindrical_edges(r_perp, stretch, coords, kd_tree):
    r_par = r_perp * stretch
    r_max = np.sqrt(r_perp**2 + r_par**2) # Bounding sphere radius

    # Spherical search to get candidate pairs
    pairs = kd_tree.query_pairs(r_max, output_type='ndarray') 
    if len(pairs) == 0:
        return np.empty((0, 2), dtype=int)
    idx_i, idx_j = pairs[:, 0], pairs[:, 1]
    pos_i = coords[idx_i]
    pos_j = coords[idx_j]
    
    # Separation vector and line-of-sight direction
    diff = pos_i - pos_j
    los = (pos_i + pos_j) # Line-of-sight vector from Earth (0,0,0)
    los_norm = np.linalg.norm(los, axis=1)
    
    # Decompose separation into parallel and perpendicular components
    dot_product = np.sum(diff * los, axis=1)
    d_par = np.abs(dot_product) / los_norm
    
    diff_sq = np.sum(diff**2, axis=1)
    # np.clip guards against floating-point rounding errors
    d_perp = np.sqrt(np.clip(diff_sq - d_par**2, 0, None)) 
    
    # Retain only pairs that satisfy the cylinder condition
    valid_mask = (d_perp <= r_perp) & (d_par <= r_par)
    return np.column_stack([idx_i[valid_mask], idx_j[valid_mask]])

### 2.2) Susceptibility and maximum fragmentation

We sweep the perpendicular linking length $r_\perp$ and, at each value, compute two independent observables of the resulting cylindrical graph.

**Susceptibility:**

$$
\chi(r_\perp) \;=\; \frac{1}{N} \sum_{s \neq \text{GCC}} s^2 \, n_s ,
$$

where the sum runs over all finite components (excluding the giant connected component), $s$ is component size, $n_s$ is the number of components of that size, and $N$ is the total number of galaxies. $\chi$ diverges at the percolation threshold $r_c$, the linking length at which an infinite cluster first spans the system. We identify $r_{\rm critical}$ as the location of the first peak of $\chi$.

**Number of valid groups:**

Since we are searching for galaxy groups, which are the smallest gravitationally bound structures (not clusters or superclusters), we seek the linking length at which the number of such groups is maximized. For each $r_\perp$, let $G_r$ denote the set of connected components. Then:

$$
N_{\rm groups}(r_\perp) \;=\; |\{\, c \in G_r  \mid  2 \leq |c| < |\text{GCC}| \,\} |.
$$

This counts connected components that are neither isolated galaxies nor part of the GCC. Its peak identifies the linking length at which the network is most fragmented into distinct groups, before those groups begin to merge into a percolating filamentary network. We adopt this $r_{\rm groups}$ as the working linking length for group identification.

**Why both?**

$r_{\rm critical}$ marks the percolation phase transition, the linking length at which clusters and superclusters begin to form. While this reflects the hierarchical structure of the network, we use $r_{\rm critical}$ only as an upper bound to ensure we remain in the sub-critical regime: $r_{\rm groups} < r_{\rm critical}$.

Since $r_{\rm groups}$ is the actual group-finding scale, we refer to it as $r_{\rm link}$.

In [ ]:
# Sweep r_perp and compute susceptibility and fragmentation count simultaneously
r_perp_values = np.arange(0.1,3 ,.02)
susceptibilities =[]
gcc_percentages = []
num_valid_groups =[]
for r_perp in r_perp_values:
    edges = get_cylindrical_edges(r_perp, stretch_factor, coordinates, tree)
    # Build temporary graph to find connected components at this linking length
    temp_graph = nx.Graph()
    temp_graph.add_nodes_from(range(number_of_galaxies))
    temp_graph.add_edges_from(edges)
    components = list(nx.connected_components(temp_graph))
    # Count components with 2 <= size < GCC size (valid groups)
    valid_groups =[c for c in components if len(c) >= 2 and len(c) < max(len(cp) for cp in components)]
    num_valid_groups.append(len(valid_groups))
    if len(components) <= 1:
        susceptibilities.append(0)
        continue
    
    gcc_size = max(len(c) for c in components)
    finite_components = [c for c in components if len(c) < gcc_size]
    gcc_percentages.append(100 * gcc_size / number_of_galaxies)

    if not finite_components:
        susceptibilities.append(0)
        continue
        
    chi = sum(len(c)**2 for c in finite_components) / number_of_galaxies
    susceptibilities.append(chi)

r_critical_perp = r_perp_values[np.argmax(susceptibilities)]
r_groups_perp = r_perp_values[np.argmax(num_valid_groups)] 

print(f"Topological Percolation Threshold (Perp): {r_critical_perp:.2f} h^-1 Mpc")
print(f"Topological Group Threshold (Perp): {r_groups_perp:.2f} h^-1 Mpc")
print(f"Implied Line-of-Sight Threshold (Par): {r_groups_perp * stretch_factor:.2f} h^-1 Mpc")

### 2.3) Reality check: the dimensionless linking parameter $b$

In the classical Friends-of-Friends literature, the linking length is reported as a dimensionless ratio:

$$
b \equiv \frac{r_{\rm link}}{\bar{d}}, \qquad \bar{d} = n^{-1/3}, \qquad
n = \frac{N_{\rm gal}}{V_{\rm shell}},
$$

where $\bar{d}$ is the mean inter-galaxy spacing and $n$ is the number density of the volume-limited sample.

Two reference ranges are relevant. Earlier papers such as **Davis et al. 1985** adopt a fixed $b \in [0.14,\, 0.20]$, while **Tempel et al. (2014)** use a redshift-dependent linking length ($b \approx 0.113$--$0.115$ for the volume-limited samples that overlap our redshift slice $0.02 \leq z \leq 0.06$).

We compute $b$ here only as post-hoc validation of our topologically derived $r_{\rm groups}$; it is not used as an input to the algorithm.

**Volume of the sample.** The volume-limited slice occupies a spherical shell between the comoving distances at $z_{\min}$ and $z_{\max}$, restricted to the SDSS footprint:

$$
V_{\rm shell} \;=\; \frac{4\pi}{3}\,(d_{\max}^{3} - d_{\min}^{3}) \,\cdot\, f_{\rm sky},
\qquad f_{\rm sky} = \frac{7221\,{\rm deg}^2}{41252.96\,{\rm deg}^2}.
$$

where $d_{\min}$ and $d_{\max}$ are the comoving distances at $z_{\min}$ and $z_{\max}$, and $f_{\rm sky}$ is the SDSS Main Galaxy Sample sky fraction of $\sim 7221\,{\rm deg}^2$ divided by the area of the full sphere.

In [7]:
SDSS_sky_fraction = 7221 / 41252.96
d_min = cosmo.comoving_distance(z_min).value 
d_max = cosmo.comoving_distance(z_max).value 
limited_volume = (4/3) * np.pi * (d_max**3 - d_min**3) * SDSS_sky_fraction
number_density = number_of_galaxies / limited_volume
d_mean = number_density**(-1/3)
b = r_groups_perp / d_mean
print(f"\nLinking parameter b = {b:.3f}")
print(f"Tempel comparison : b ≈ 0.113–0.115")
print(f"Classical FoF range (Davis 1985, Berlind 2006): 0.14 < b < 0.20")


Linking parameter b = 0.144
Tempel comparison : b ≈ 0.113–0.115
Classical FoF range (Davis 1985, Berlind 2006): 0.14 < b < 0.20


### Giant connected component (GCC) diagnostic

The GCC is the single largest connected structure in the network. As $r_\perp$ increases, the network undergoes a percolation transition in which small, isolated galaxy groups merge into a single large structure spanning the sampled volume.

One approach is to characterize this transition by the derivative of the GCC fraction with respect to $r_\perp$ ($\frac{d(\text{GCC})}{d r_\perp}$). The point at which the GCC grows most rapidly (the maximum gradient) is a classical heuristic for estimating the percolation threshold. Alternatively, the percolation threshold can be identified as the location of the susceptibility peak.

The present problem, however, requires more care. The galaxy network contains structures at multiple scales: galaxy groups (the targets of this method), clusters, and superclusters, each with distinct physical properties.

Both the susceptibility and the maximum gradient methods identify the linking length at which clusters and superclusters form, not the scale of individual groups. We therefore adopt the maximum fragmentation criterion: the network is most fragmented into distinct finite groups at the linking length where the number of such groups is maximized. The corresponding $r_\perp$ is adopted as the linking length for group identification.

By plotting the GCC coverage, we can visually diagnose the phase transition and compare this inflection point against our explicitly defined topological markers:
- **Maximum fragmentation** ($r_{\rm groups}$): the linking length at which isolated groups are captured before they merge.
- **Susceptibility peak** ($r_{\rm critical}$): the true topological percolation threshold.
- **Max gradient location**: the steepest growth phase of the supercluster network (shown for reference only).

In [ ]:
gcc_array = np.array(gcc_percentages)

# Differentiate the GCC curve to locate the maximum growth rate
dgcc = np.gradient(gcc_array, r_perp_values)
r_max_gradient = r_perp_values[np.argmax(dgcc)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
fig.suptitle(r'GCC Diagnostic: Cylindrical $r_\perp$ Scan', fontsize=14)

# Top panel: GCC coverage 
ax1.plot(r_perp_values, gcc_array, marker='o', markersize=4,
         color='#45A29E', linewidth=2, label='GCC coverage')
ax1.axvline(x=r_max_gradient, color='orange', linestyle='--',
            label=f'Max gradient ({r_max_gradient:.2f} $h^{{-1}}$ Mpc)')
ax1.axvline(x=r_critical_perp, color="#379BFF", linestyle='-.',
            label=f'Susceptibility peak ({r_critical_perp:.2f} $h^{{-1}}$ Mpc)')
ax1.axvline(x=r_groups_perp, color='#FB2E00', linestyle=':',
            label=f'Max fragmentation ({r_groups_perp:.2f} $h^{{-1}}$ Mpc)')
ax1.set_ylabel('GCC Size (% of galaxies)', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Bottom panel: gradient of GCC 
ax2.plot(r_perp_values, dgcc, marker='o', markersize=4,
         color='#C5C6C7', linewidth=2, label=r'd(GCC)/d($r_{\perp}$)')
ax2.axvline(x=r_max_gradient, color='orange', linestyle='--')
ax2.axvline(x=r_critical_perp, color="#379BFF", linestyle='-.')
ax2.axvline(x=r_groups_perp, color='#FB2E00', linestyle=':')
ax2.set_xlabel(r'Transverse Linking Length $r_{\perp}$ ($h^{-1}$ Mpc)', fontsize=12)
ax2.set_ylabel(r'd(GCC) / d($r_{\perp}$)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max gradient r_c:      {r_max_gradient:.2f} h^-1 Mpc")
print(f"Susceptibility peak (r_critical):    {r_critical_perp:.2f} h^-1 Mpc")
print(f"Max fragmentation (r_groups):        {r_groups_perp:.2f} h^-1 Mpc")

### 2.4 Building the final graph

With $r_{\rm groups}^\perp$ found by the maximum fragmentation, we build the final graph using the same cylindrical FoF method we used before, run once at the chosen linking length.

In [ ]:
# Build the final network at r_groups_perp
final_edges = get_cylindrical_edges(r_groups_perp, stretch_factor, coordinates, tree)
G = nx.Graph()
G.add_nodes_from(range(number_of_galaxies))
G.add_edges_from(final_edges)

print(f"Network built. Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

### 2.4) Forming galaxy groups

In the cylindrical FoF method, **connected components of the linking graph are the galaxy groups**. Two galaxies belong to the same group if a chain of pairwise linkings connects them (the friends-of-friends rule).

Each connected component is assigned a unique integer, added as a column in the `mgs` dataframe. After this step, each galaxy carries:

- `GroupID`: Tempel's reference label. NaN for field galaxies (single galaxies not belonging to any group).
- `Detected_GroupID`: our graph-derived label (assigned to all galaxies; field galaxies receive their own distinct ID).

In [ ]:
# Connected components are the groups by definition
groups = list(nx.connected_components(G))
group_labels = np.zeros(len(mgs), dtype=int)
for group_id, members in enumerate(groups):
    for node in members:
        group_labels[node] = group_id

mgs['Detected_GroupID'] = group_labels

# Phase 3: evaluation

### 3.1 Completeness and purity by richness bin

For each group in Tempel's catalog ($N \geq 2$),
we find the detected component with the largest membership overlap and
compute two metrics:

$$
\text{Completeness} = \frac{|T \cap D|}{|T|}, \qquad
\text{Purity} = \frac{|T \cap D|}{|D|},
$$

where $T$ is the set of Tempel members and $D$ is the set of members
in the best-matching detected group. Completeness measures what fraction
of a true group was recovered; purity measures what fraction of the
detected group genuinely belongs to that group.

We report the mean of both metrics in richness bins. The expected
behaviour is high completeness across all bins (the cylinder captures
most members) with purity declining at high $N$ (rich clusters have
large velocity dispersions that the fixed stretch factor cannot fully accommodate).

In [ ]:
# For each Tempel group, find the detected component with the greatest membership overlap
# Completeness: fraction of true group members recovered
# Purity: fraction of detected group members that are genuine matches

results = []
for true_id in mgs['GroupID'].dropna().unique():
    true_members = set(mgs[mgs['GroupID'] == true_id].index)
    if len(true_members) < 2:
        continue
    
    # Identify the detected group with the most overlap
    detected_ids = mgs.loc[list(true_members), 'Detected_GroupID']
    best_detected = detected_ids.value_counts().index[0]
    detected_members = set(mgs[mgs['Detected_GroupID'] == best_detected].index)
    
    overlap = len(true_members & detected_members)
    completeness = overlap / len(true_members)
    purity = overlap / len(detected_members)
    
    results.append({
        'True_GroupID': true_id,
        'N_true': len(true_members),
        'Completeness': completeness,
        'Purity': purity
    })

validation_df = pd.DataFrame(results)
print(validation_df.groupby(pd.cut(validation_df['N_true'], 
      bins=[1,2,5,10,50,500]))[['Completeness','Purity']].mean())

### 3.2 Visual cross-check across the redshift slice

To assess whether the group catalogue behaves uniformly across the volume and whether $r_{\rm groups}^\perp$ remains a reasonable linking length at all redshifts in the slice, we project the RA ($\alpha$)--Dec ($\delta$) plane in four redshift bins of width $\Delta z = 0.01$ and plot the detected groups alongside Tempel's reference.

Each column corresponds to one redshift bin. Grouped galaxies are coloured by `Detected_GroupID` (top) or Tempel `GroupID` (bottom), with marker size scaled by group richness; field galaxies are shown in grey.

In [ ]:
def make_group_colors(group_ids, seed=0):
    """Stable shuffled qualitative colour map."""
    rng = np.random.default_rng(seed)
    base = plt.cm.tab20(np.linspace(0, 1, 20))
    unique = np.asarray(sorted(group_ids))
    perm = rng.permutation(len(unique))
    return {gid: base[perm[i] % 20] for i, gid in enumerate(unique)}


def plot_radec_panel(ax, mgs, group_col, z_lo, z_hi, *, panel_title):
    """One RA-Dec panel, sliced in redshift."""
    slab = mgs[mgs['z'].between(z_lo, z_hi, inclusive='left')]

    counts = slab[group_col].value_counts(dropna=True)
    grouped_ids = counts[counts > 1].index
    is_grouped = slab[group_col].isin(grouped_ids) & slab[group_col].notna()

    grouped = slab[is_grouped]
    isolated = slab[~is_grouped]

    # background of singletons / field galaxies
    ax.scatter(isolated['ra'], isolated['dec'],
               c='grey', s=1, alpha=0.2, label='Isolated / field')

    # grouped galaxies, coloured and sized by richness
    if len(grouped) > 0:
        richness = grouped[group_col].map(counts).astype(float)
        sizes = (5.0 + 1.5 * np.log10(richness)) ** 2
        cmap = make_group_colors(grouped_ids)
        colors = np.array([cmap[g] for g in grouped[group_col].values])
        ax.scatter(grouped['ra'], grouped['dec'],
                   c=colors, s=sizes, alpha=0.85, edgecolors='none',
                   label='Grouped')

    ax.set_xlabel('RA [deg]')
    ax.set_ylabel('Dec [deg]')
    ax.set_aspect('equal')
    ax.set_title(panel_title, fontsize=11)
    return len(slab), len(grouped)


# --- main figure ---------------------------------------------------------

z_edges = np.arange(z_min, z_max + 1e-9, 0.01)   # [0.02, 0.03, 0.04, 0.05, 0.06]
n_bins = len(z_edges) - 1                          # 4 bins

# Adjusted figsize to match a 2-row x 4-column layout with equal aspect ratio plots.
fig, axes = plt.subplots(2, n_bins,
                         figsize=(24, 10),
                         constrained_layout=True,
                         sharex=True, sharey=True)

# Ensure axes is always 2D, even if n_bins == 1.
if axes.ndim == 1:
    axes = axes[np.newaxis, :]

for col, (z_lo, z_hi) in enumerate(zip(z_edges[:-1], z_edges[1:])):

    n_slab, n_det = plot_radec_panel(
        axes[0, col], mgs,
        group_col='Detected_GroupID',
        z_lo=z_lo, z_hi=z_hi,
        panel_title=(f'Detected-  '
                     f'($z \\in [{z_lo:.2f}, {z_hi:.2f})$, '
                     f'$N_{{\\rm gal}} = {n_slab if False else 0}$)'),
    )
    plot_radec_panel(
        axes[1, col], mgs,
        group_col='GroupID',
        z_lo=z_lo, z_hi=z_hi,
        panel_title=(f'Tempel et al. 2014 - '
                     f'($z \\in [{z_lo:.2f}, {z_hi:.2f})$)'),
    )

    # Update the top-panel title once n_slab and n_det are known.
    axes[0, col].set_title(
        f'Detected ($z \\in [{z_lo:.2f}, {z_hi:.2f})$, '
        f'$N_{{\\rm gal}} = {n_slab}$, $N_{{\\rm grouped}} = {n_det}$)',
        fontsize=11,
    )

fig.suptitle(
    'Galaxy distribution and groups in (RA, Dec): '
    'detected (top) vs Tempel et al. 2014 (bottom), per redshift bin',
    fontsize=14
)
plt.show()

### 3.3 Group-richness distribution

A side-by-side comparison of group sizes between the detected catalogue and Tempel et al. (2014). For each catalogue we count, per group, the number of member galaxies, then plot a log-log histogram of those counts.

Both histograms are restricted to groups of richness $N \geq 2$, so single-galaxy components (field galaxies) are excluded from the comparison. Two systematic failure modes are visible in such a comparison:

- **Shift to higher $N$:** indicates over-merging, in which small groups have merged into larger structures because the linking cylinder is too wide.

- **Shift to lower $N$:** indicates under-merging and fragmentation, in which rich clusters are split because the cylinder is too narrow or the stretch factor is too short to absorb their full Fingers-of-God extent.

In [ ]:
detected_sizes = mgs['Detected_GroupID'].value_counts()
detected_sizes = detected_sizes[detected_sizes >= 2] # Restrict to groups with at least 2 members

tempel_sizes = mgs['GroupID'].value_counts()
tempel_sizes = tempel_sizes[tempel_sizes >= 2] # Restrict to groups with at least 2 members

fig, ax = plt.subplots(figsize=(8, 6))
N_max = max(detected_sizes.max(), tempel_sizes.max())
bins = np.logspace(np.log10(2), np.log10(N_max)+0.1, 25)
ax.hist(detected_sizes, bins=bins, alpha=.7, histtype='stepfilled', label='Detected (this project)', color='steelblue')
ax.hist(tempel_sizes, bins=bins, histtype='step', label='Tempel et al. 2014', color='red')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Group Richness N', fontsize=12)
ax.set_ylabel('Number of Groups', fontsize=12)
ax.set_title('Group Richness Distribution', fontsize=12)
ax.legend()
plt.show()

### 3.4 The phase-transition figure

This figure presents the two independent topological observables computed during the percolation scan, plotted on a shared $r_\perp$ axis:

- **Susceptibility $\chi(r_\perp)$** peaks at the percolation threshold $r_{\rm critical}^\perp$. Its peak marks the linking length at which an infinite cluster first spans the system.
- **Fragmentation count $N_{\rm groups}(r_\perp)$** counts finite components of size $2 \leq s < |\text{GCC}|$. Its peak marks the linking length at which the network is most fragmented into distinct, finite groups, before merging into the percolating phase.

Two vertical lines mark $r_{\rm critical}^\perp$ and $r_{\rm groups}^\perp$. The defining property of the method, which distinguishes $r_{\rm groups}^\perp$ as a group scale rather than a cluster scale, is the inequality

$$
r_{\rm groups}^\perp < r_{\rm critical}^\perp.
$$

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot susceptibility
ax1.plot(r_perp_values, susceptibilities, marker='o', linestyle='-', color="#35AAA4", linewidth=2, label=r'Susceptibility ($\chi$)')

# Mark the critical points
ax1.axvline(x=r_critical_perp, color="#00211F", linestyle='--', label=f'Percolation Critical ({r_critical_perp:.2f} $h^{{-1}}$ Mpc)')
ax1.axvline(x=r_groups_perp, color="#FB2E00", linestyle=':', label=f'Maximum Fragmentation ({r_groups_perp:.2f} $h^{{-1}}$ Mpc)')

ax1.set_title(r"Anisotropic Phase Transition (Cylindrical $r_{\perp}$)", fontsize=14, color='black')
ax1.set_xlabel(r"Transverse Linking Length $r_{\perp}$ ($h^{-1}$ Mpc)", fontsize=12, color='black')
ax1.set_ylabel("Susceptibility (excluding GCC)", fontsize=12, color='black')
ax1.grid(color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

# Add a twin axis to show the number of groups (maximum fragmentation)
ax2 = ax1.twinx()
ax2.plot(r_perp_values, num_valid_groups, marker='s', linestyle='-', color="#2CB5FF", linewidth=1, alpha=0.7, label='Number of Groups')
ax2.set_ylabel("Number of Isolated Groups", fontsize=12, color='black')

# Combine legends
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

plt.tight_layout()
plt.show()